# Our RSGM against the pretrained PoseD-Flow, on their data (19 joints)


A quick comparison: the prior of PoseD-Flow (Nadar, Foti, Birdal, CVPR 2026), their released checkpoint, against our RSGM trained on **their** AMASS (the preprocessing of DPoser, converted by `scripts/preprocess/convert_posedflow.py`, split `posedflow`).

- **Joints.** Their model is on SO(3)^21 (SMPL joints 1–21), ours on 19 (without the toes). Their samples are cut to our 19 joints (`hp/models/posedflow.py`). The toes are leaves: they move no other joint.
- **Data.** Train: every 10th of their 18.6M training poses (1.86M). Test: all 54,200 test poses. Their checkpoint saw the whole train set; ours the 10% subset (neighbouring frames at 30 Hz are nearly the same).
- **Metrics.**
  1. our harness (`hp.evaluation.evaluate`): Wasserstein, recall, precision against the test poses, with the training poses as the yardstick;
  2. the protocol of their paper (Tab. 1; `hp/evaluation/posedflow_protocol.py`), on 20 sets × 500 samples, mean ± 95% CI:
     - **FID**: Fréchet distance of the 22 SMPL joint positions (SMPL neutral v1.1.0, betas = 0) against their precomputed statistics `gt_mean_var_v2.npz`. Also against the statistics of our train subset with the same skeleton ("FID own"), which is self-consistent: their statistics sit ~0.005 away in the mean, likely another SMPL file;
     - **APD**: mean pairwise joint distance, cm;
     - **dNN**: distance to the nearest training pose. Theirs is on 21 joints against all 18.6M training poses. Here it's on 19 joints against the 1.86M of the subset, so it's larger than in their table, for both models alike.

  Their checkpoint reproduces their table here: FID 0.014 and APD 15.5 cm, against 0.013–0.014 and 15.4–15.5 in the paper.
- **Likelihood.** It is not comparable. A CNF density on SO(3)^21 cannot be marginalized to 19 joints, so their model has no log-likelihood on our joints. RSGM's likelihood is printed for the record.
- **Their sampler.** The paper uses midpoint with 100 steps and their script Euler with 1000. Both are evaluated. Their script also samples with the raw weights, not the EMA ones, because it never calls `eval()`. dNN differs by 0.0016 rad between the two (a quick check).

In [ ]:
import json
import re
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import numpy as np
import torch

from hp.evaluation.data import load_reference
from hp.evaluation.evaluate import EvalConfig, evaluate
from hp.evaluation.posedflow_protocol import SmplSkeleton, apd, confidence, fid, nearest_neighbour_distance
from hp.models.io import MANIFEST, load_model
from hp.models.posedflow import PoseDFlowPretrained

assert load_model.__module__ == "hp.models.io" and sys.modules["hp.models.io"].__file__.startswith(SRC), \
    f"hp comes from {sys.modules['hp.models.io'].__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_posedflow_c10"  # convert_posedflow.py --train-stride 10 --valid-stride 100
CHECKPOINT = "~/data/posedflow_ckpt/checkpoints/epoch-009_step-0_loss-0.0000.ckpt"
SKELETON = "~/data/body_models/smpl/smpl_neutral_skeleton.npz"  # scripts/preprocess/convert_smpl_skeleton.py
GT_STATS = "~/Documents/PoseDFlow/examples/gt_mean_var_v2.npz"  # their FID statistics (muj, covj)

# Any number of models; comment a line out to leave a model out. Two kinds:
#   posedflow=dict(weights=, method=, sample_steps=)   their checkpoint (hp/models/posedflow.py)
#   run=<run folder, a folder of runs (the newest finished one), or a checkpoint folder>, or a LIST of them (e.g.
#       seeds): each is evaluated and cached by itself, and the tables add their mean +- std over the runs;
#       optional weights= (default weights_ema), sample_steps=, likelihood_steps=, eps= (set on the loaded model)
MODELS = {
    "PoseD-Flow (midpoint 100)": dict(posedflow=dict(weights="ema", method="midpoint", sample_steps=100)),
    "PoseD-Flow (Euler 1000, their script)": dict(posedflow=dict(weights="raw", method="euler", sample_steps=1000)),
    "RSGM t23, our AMASS": dict(run="~/data/experiments/rsgm_tuned_t23"),
    # DPoser-body (hp/models/dposer.py) trained from scratch on ALL their training poses (workstation1, --config paper)
    "DPoser, their full data": dict(run="~/data/experiments/full/dposer_posedflow_s10"),
    # the same weights with 100 Euler-Maruyama steps instead of their 1000: the sampling cost of RSGM and PoseD-Flow
    "DPoser, their full data, 100 steps": dict(run="~/data/experiments/full/dposer_posedflow_s10", sample_steps=100),
    # DPoser with the budget of the RSGM runs: 30k steps of 4096 poses on every 10th training pose (--config rsgm_budget)
    "DPoser, RSGM budget": dict(run="~/data/experiments/dposer_posedflow_rsgm_budget_s10"),
    "DPoser, RSGM budget, 100 steps": dict(run="~/data/experiments/dposer_posedflow_rsgm_budget_s10", sample_steps=100),
    # the small RSGM on ALL their training poses (18.6M instead of every 10th), the same 30k steps of 4096
    "RSGM 512x4, their full data": dict(run="~/data/experiments/full/rsgm_posedflow_small_s10"),
    "RSGM 1536x6, their data": dict(run="~/data/experiments/rsgm_posedflow_s10"),
    # 101.6M parameters, lr 3e-4 with 1000 steps of warm-up and a cosine schedule (bigmama); the validation loss is
    # lowest at 8k steps (5.91) and rises to 6.21 at 30k: the final weights and the best checkpoint
    "RSGM 4096x7, their data": dict(run="~/data/experiments/rsgm_posedflow_large_s10"),
    "RSGM 4096x7, their data, best (8k)": dict(run="~/data/experiments/rsgm_posedflow_large_s10/run_0001/checkpoint_008000"),
    "RSGM 512x4, their data": dict(run=[f"~/data/experiments/rsgm_posedflow_small_s{seed}" for seed in (10, 11, 12)]),
    "RSGM 512x4, their data, 300 steps": dict(run="~/data/experiments/rsgm_posedflow_small_s10", sample_steps=300),
    "RSGM 512x4, their data, 1000 steps": dict(run="~/data/experiments/rsgm_posedflow_small_s10", sample_steps=1000),
    # the sampler stops at eps = 2e-4, below the times the loss was trained on (loss_t_min = 2e-3)
    "RSGM 512x4, their data, eps 2e-3": dict(run="~/data/experiments/rsgm_posedflow_small_s10", eps=2e-3),
    "RSGM 512x4, their data, eps 1e-2": dict(run="~/data/experiments/rsgm_posedflow_small_s10", eps=1e-2),
    "RSGM 512x4, their data, raw weights": dict(run="~/data/experiments/rsgm_posedflow_small_s10", weights="weights"),
    # "RSGM 1536x6, their data, 1000 steps": dict(run="~/data/experiments/rsgm_posedflow_s10", sample_steps=1000),
}

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
HARNESS = EvalConfig()  # 20k samples, 2000 queries, 3 repeats
N_CONTRAST = 100_000  # training poses as the yardstick of the harness
SETS, SET_SIZE = 20, 500  # as their paper
CACHE = Path("~/data/experiments/posedflow_comparison/cache").expanduser()  # cache/<model>/<stage>.json
RECOMPUTE = set()  # names of models (or of their runs, "<name> / <run>") to evaluate again; True: all
SHOW_RUNS = True  # the rows of the single runs of a model with several runs, besides their mean
# --------------------------------------------------------------------------------------
CACHE.mkdir(parents=True, exist_ok=True)


def run_folder(path):
    path = Path(path).expanduser()
    if (path / MANIFEST).is_file():
        return path
    runs = sorted(p for p in path.glob("run_[0-9][0-9][0-9][0-9]") if (p / "history.json").is_file() and (p / MANIFEST).is_file())
    return runs[-1] if runs else None

## 1. The data

In [ ]:
test = load_reference("test", ROTATIONS, "posedflow")
train = load_reference("train", ROTATIONS, "posedflow")
contrast = train[torch.from_numpy(np.random.default_rng(0).choice(len(train), N_CONTRAST, replace=False))]
print(f"test {len(test):,}   train {len(train):,}   contrast {len(contrast):,}")

## 2. The models


Every entry of `MODELS` is built from its spec:
- **PoseD-Flow**: their checkpoint, trained and evaluated on their data.
- **RSGM runs**, which may come from different data:
  - *their data*: trained on their train set (`explore/training/rsgm_posedflow_train.ipynb`). This is the fair comparison;
  - *our AMASS*: trained on our AMASS, the SMPL-X G fit at all frames. Theirs is a SMPL fit at 30 Hz with the first and last 10% of every sequence cut. The datasets of the split are the same (the VPoser split: test Transitions + SSM). Such a model is evaluated on data it has never seen, so the comparison favours PoseD-Flow.

**The cache.** Every model has its own folder `cache/<name>/` with one file per stage (`harness.json`, `paper.json`). A file also stores what it was computed from: the spec, the run folder, the size and time of the weights file, and the settings of the evaluation. A saved result is reused only if all of these match, so a retrained run or a changed setting is evaluated again by itself. Renaming a model starts a new folder.

In [ ]:
def resolve(spec, path):
    """(builder, identity) of one model: how to build it, and what its results depend on (for the cache)."""
    if "posedflow" in spec:
        args = dict(spec["posedflow"])
        checkpoint = Path(CHECKPOINT).expanduser()
        identity = {"kind": "posedflow", "args": args, "checkpoint": str(checkpoint), "mtime": checkpoint.stat().st_mtime}
        return (lambda: PoseDFlowPretrained(checkpoint, **args).to(DEVICE)), identity
    run = run_folder(path)
    if run is None:
        return None, None
    weights = spec.get("weights", "weights_ema")
    file = run / f"{weights}.pt"
    overrides = {k: spec[k] for k in ("sample_steps", "likelihood_steps", "eps") if k in spec}
    identity = {"kind": "run", "run": str(run), "weights": weights, "size": file.stat().st_size,
                "mtime": file.stat().st_mtime, "overrides": overrides}

    def build():
        model = load_model(run, device=DEVICE, weights=weights)
        for k, v in overrides.items():
            setattr(model, k, v)
        return model
    return build, identity


def tag(path):
    """A short name of a run: its experiment folder (rsgm_posedflow_small_s11), or the checkpoint folder."""
    p = Path(path).expanduser()
    return p.parent.name if re.fullmatch(r"run_\d{4}", p.name) else p.name


def members(name, spec):
    """[(label, path)]: a model with one run keeps its name (and its cache); several runs get "<name> / <run>"."""
    if "posedflow" in spec:
        return [(name, None)]
    paths = spec["run"] if isinstance(spec["run"], (list, tuple)) else [spec["run"]]
    return [(name, paths[0])] if len(paths) == 1 else [(f"{name} / {tag(p)}", p) for p in paths]


builders, identities, groups, group_of = {}, {}, {}, {}
for name, spec in MODELS.items():
    for label, path in members(name, spec):
        build, identity = resolve(spec, path)
        if build is None:
            print(f"{label}: skipped, no finished run in {path}")
            continue
        builders[label], identities[label], group_of[label] = build, identity, name
        groups.setdefault(name, []).append(label)
        print(f"{label}: {identity.get('run', identity.get('checkpoint'))}")


def slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


def portable(value):
    """Settings as compared across machines (e.g. results computed on a GPU server by
    scripts/evaluate_posedflow_comparison.py): a home folder (/home/<user>, /Users/<user>) as ~, file times in whole
    seconds."""
    if isinstance(value, dict):
        return {k: (int(v) if k == "mtime" else portable(v)) for k, v in value.items()}
    if isinstance(value, list):
        return [portable(v) for v in value]
    return re.sub(r"^/(home|Users)/[^/]+(?=/|$)", "~", value) if isinstance(value, str) else value


def same_elsewhere(stage, settings):
    """A result of `stage` with exactly these settings saved under another name (e.g. before a rename), or None."""
    for other in sorted(CACHE.glob(f"*/{stage}.json")):
        saved = json.loads(other.read_text())
        if portable(saved.get("settings")) == portable(settings):
            return saved["result"], other.parent.name
    return None, None


def cached(label, stage, settings, compute):
    """The result of `stage` for `label` from cache/<label>/<stage>.json if it was computed with these settings (or
    from another folder with the same settings, then copied); otherwise computed and saved."""
    path = CACHE / slug(label) / f"{stage}.json"
    settings = json.loads(json.dumps(settings))  # as it reads back from json
    forced = RECOMPUTE is True or label in RECOMPUTE or group_of.get(label) in RECOMPUTE
    result = None
    if not forced:
        if path.is_file():
            saved = json.loads(path.read_text())
            if portable(saved.get("settings")) == portable(settings):
                return saved["result"]
            print(f"  {label} / {stage}: the cached result is of other settings")
        result, source = same_elsewhere(stage, settings)
        if result is not None:
            print(f"  {label} / {stage}: the same evaluation is cached as {source}, reused")
    if result is None:
        result = compute()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({"name": label, "stage": stage, "settings": settings, "result": result}, indent=1))
    return result


def rows(results, extra=()):
    """(row name, [results]) in the order of MODELS: one result per row, or the results of all runs of a model for its
    mean row (preceded by the rows of its runs if SHOW_RUNS); `extra` rows (real poses) come first."""
    out = [(name, [results[name]]) for name in extra]
    for name, labels in groups.items():
        if len(labels) == 1:
            out.append((name, [results[labels[0]]]))
            continue
        if SHOW_RUNS:
            out += [(f"  {label.split(' / ', 1)[1]}", [results[label]]) for label in labels]
        out.append((f"{name} ({len(labels)} runs)", [results[label] for label in labels]))
    return out


def over_runs(values):
    """Mean and std (ddof 1) over the runs."""
    v = np.asarray(values, dtype=np.float64)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else float("nan"))

## 3. Our harness: Wasserstein, recall, precision

In [ ]:
harness_settings = {"config": HARNESS.__dict__, "rotations": ROTATIONS, "test": len(test), "contrast": N_CONTRAST}
harness = {}
for label, build in builders.items():
    harness[label] = cached(label, "harness", {"model": identities[label], **harness_settings},
                            lambda build=build: evaluate(build(), test, HARNESS, contrast=contrast, progress=True))
    print(f"--- {label} done")

In [ ]:
HARNESS_COLUMNS = {
    "Wasserstein": lambda r: r["wasserstein"]["generated"]["joint_mean_deg"],
    "recall": lambda r: r["recall"]["mean_deg"],
    "precision": lambda r: r["precision"]["mean_deg"],
}


def harness_cells(results):
    """One run: mean ± std over the repeats of the evaluation; several: mean ± std of their means over the runs."""
    cells = []
    for get in HARNESS_COLUMNS.values():
        if len(results) == 1:
            m, s = get(results[0])["mean"], get(results[0])["std"]
        else:
            m, s = over_runs([get(r)["mean"] for r in results])
        cells.append(f"{m:6.2f} ± {s:4.2f}")
    lls = [r["likelihood"] for r in results]
    if lls[0] is None:
        ll = "—"
    elif len(lls) == 1:
        ll = f"{lls[0]['mean']:.2f} ({lls[0]['space']})"
    else:
        m, s = over_runs([x["mean"] for x in lls])
        ll = f"{m:.2f} ± {s:.2f} ({lls[0]['space']})"
    return cells, ll


print(f"{'':48s} " + " ".join(f"{c:>14s}" for c in HARNESS_COLUMNS) + "   log-lik. (space)")
for name, results in rows(harness):
    cells, ll = harness_cells(results)
    print(f"{name:48s} " + " ".join(f"{c:>14s}" for c in cells) + f"   {ll}")
r = next(iter(harness.values()))  # the yardstick is the same for every model
yard = [r["wasserstein"]["contrast"]["joint_mean_deg"], r["contrast_recall"]["mean_deg"], r["contrast_precision"]["mean_deg"]]
print(f"{'real train poses (yardstick)':48s} " + " ".join(f"{d['mean']:6.2f} ± {d['std']:4.2f}".rjust(14) for d in yard))
print("degrees per joint (mean over the joints); lower is better. One run: ± std over the repeats of the evaluation;\n"
      "\"(n runs)\": mean ± std over the runs (seeds) of their means.")

## 4. The protocol of their paper: FID, APD, dNN







The same 20 sets of 500 poses per model for all three metrics. Their Tab. 1 for reference (21 joints, dNN against all their training poses):

| | FID ↓ | APD (cm) ↑ | dNN (rad) ↓ |
|---|---|---|---|
| VPoser | 0.048 ± .002 | 14.684 ± .138 | 0.074 ± .000 |
| DPoser | 0.019 ± .001 | 14.992 ± .123 | 0.073 ± .001 |
| PoseFM | 0.016 ± .001 | 14.762 ± .176 | 0.079 ± .000 |
| PoseRFM (N=100) | 0.014 ± .001 | 15.481 ± .124 | 0.069 ± .001 |
| PoseRFM (N=1000) | 0.013 ± .001 | 15.544 ± .167 | 0.070 ± .001 |

In [ ]:
skeleton = SmplSkeleton(SKELETON)
gt = np.load(Path(GT_STATS).expanduser())
muj, covj = gt["muj"].astype(np.float64), gt["covj"].astype(np.float64)
own = skeleton.joints(train).reshape(len(train), -1).numpy()  # the statistics of our train subset, same skeleton
mu_own, cov_own = own.mean(0), np.cov(own, rowvar=False)
del own
train_dev = train.to(DEVICE)


def protocol(make_set):
    """FID (their statistics and ours), APD and dNN over SETS sets of SET_SIZE poses [SET_SIZE, 19, 3, 3]."""
    out = {"FID": [], "FID own": [], "APD": [], "dNN": []}
    for k in range(SETS):
        poses = make_set(k)
        joints = skeleton.joints(poses)
        out["FID"].append(fid(joints, muj, covj))
        out["FID own"].append(fid(joints, mu_own, cov_own))
        out["APD"].append(apd(joints))
        out["dNN"].append(nearest_neighbour_distance(poses, train_dev).mean().item())
    return {name: confidence(values) for name, values in out.items()}


paper_settings = {"sets": SETS, "size": SET_SIZE, "rotations": ROTATIONS, "train": len(train), "skeleton": SKELETON,
                  "gt_stats": GT_STATS}


def real(poses, seed):
    rng = np.random.default_rng(seed)
    return lambda k: poses[torch.from_numpy(rng.choice(len(poses), SET_SIZE, replace=False))]


def generated(build):
    def compute():
        model = build().eval()
        torch.manual_seed(0)
        return protocol(lambda k: model.sample(SET_SIZE).float().cpu())
    return compute


paper = {
    "real test poses": cached("real test poses", "paper", {"test": len(test), **paper_settings}, lambda: protocol(real(test, 1))),
    "real train poses": cached("real train poses", "paper", paper_settings, lambda: protocol(real(train, 2))),
}
for label, build in builders.items():
    paper[label] = cached(label, "paper", {"model": identities[label], **paper_settings}, generated(build))

METRICS = ("FID", "FID own", "APD", "dNN")


def paper_cells(results):
    """One run: mean ± 95% CI over the 20 sets; several: mean ± std of their means over the runs."""
    cells = []
    for m in METRICS:
        if len(results) == 1:
            mean, spread = results[0][m]["mean"], results[0][m]["ci95"]
        else:
            mean, spread = over_runs([r[m]["mean"] for r in results])
        cells.append(f"{mean:.2f} ± {spread:.2f}" if m == "APD" else f"{mean:.4f} ± {spread:.4f}")
    return cells


print(f"{'':48s} {'FID':>16s} {'FID own':>16s} {'APD (cm)':>16s} {'dNN (rad)':>16s}")
for name, results in rows(paper, extra=("real test poses", "real train poses")):
    print(f"{name:48s} " + " ".join(f"{c:>16s}" for c in paper_cells(results)))
print("One run: ± 95% CI over the sets; \"(n runs)\": mean ± std over the runs (seeds) of their means.")

## 5. Reading

- FID of real test poses is not 0: the test subjects differ from the training ones that the statistics come from, and 500 poses give a biased covariance. The real train poses are what the statistics describe.
- The real train poses are the poses the statistics describe: their FID is the floor of 500-pose sets (about 0.011), and their dNN is 0 by construction (each is its own nearest neighbour).
- dNN alone rewards memorization: a model that returns training poses has dNN 0. Read it with recall and precision. Real unseen test poses give the scale of a good value.
- Recall (real → nearest generated) shows coverage; precision (generated → nearest real) shows realism. The contrast row is what real poses of other subjects score.